# Analysis Dataset

This notebook builds the tables used by the analysis notebooks from the results of the experiment.

The process consists of:
1. **Model Runs**: Listing the model runs of the study and their characteristics.
2. **Inferences**: Reading the per-prompt records of every run.
3. **Inference Flags**: Marking truncated, cut-off and correct responses.
4. **Repetitions**: Joining the run table with the counts of each repetition.
5. **Questions**: Summarizing the repetitions of each question.
6. **Model Run Summary**: Summarizing the repetitions of each model run.
7. **Validation**: Checking the tables against the run table.
8. **Saving**: Writing the tables to `analysis/data/`.

### Setup & Imports
We import the libraries for data manipulation (`pandas`, `numpy`) and the paths shared by the analysis notebooks (`common.py`).

In [1]:
import json
import re

import numpy as np
import pandas as pd

from common import DATA_DIR, QUESTIONS, REPETITIONS, RESULTS_DIRS, ROOT_DIR, load_run_table

# Limit of generated tokens of a run: 2,048 tokens, plus a thinking budget of 4,096 tokens for reasoning models
MAX_NEW_TOKENS = 2048
MAX_THINKING_TOKENS = 4096

### 1. Model Runs
We read the selected models and derive the characteristics of each model run (family, size category, operating mode, architecture). The three Ministral 3 instruct models that ran with 8-bit weights in the main execution are added as separate runs, marked as not being part of the study (`in_study = False`).

In [2]:
FAMILIES = {
    "meta-llama": "Llama",
    "Qwen": "Qwen",
    "google": "Gemma",
    "deepseek-ai": "DeepSeek",
    "microsoft": "Phi",
    "mistralai": "Mistral",
}

selected = pd.read_csv(ROOT_DIR / "model-selection/selected_models.csv")

model_runs = pd.DataFrame({
    "model_run": selected["model"],
    "model": selected["model"].str.replace(r" \(.* Mode\)$", "", regex=True),
    "family": selected["model"].str.split("/").str[0].map(FAMILIES),
    "release": selected["collection_name"],
    "release_date": pd.to_datetime(selected["release_date"]),
    "category": selected["category"],
    "mode": np.where(selected["is_thinking"], "Reasoning", "Instruct"),
    "dual_mode": selected["supports_enable_thinking"],
    "architecture": np.where(selected["is_moe"], "Sparsely activated", "Dense"),
    "total_parameters": selected["total_parameters"],
    "activated_parameters": selected["activated_parameters"],
    "precision": "16-bit",
    "in_study": True,
})

# 8-bit runs of the main execution, replaced in the selection by their 16-bit versions
superseded = model_runs[model_runs["model"].str.endswith("-BF16")].copy()
superseded["model"] = superseded["model"].str.removesuffix("-BF16")
superseded["model_run"] = superseded["model"]
superseded["precision"] = "8-bit"
superseded["in_study"] = False
model_runs = pd.concat([model_runs, superseded], ignore_index=True)

print(f"Model runs of the study: {model_runs['in_study'].sum()}   models: {model_runs.loc[model_runs['in_study'], 'model'].nunique()}")
print(f"Runs kept for comparison only: {(~model_runs['in_study']).sum()}")
model_runs.groupby(["category", "mode"])["in_study"].sum().unstack()

Model runs of the study: 53   models: 47
Runs kept for comparison only: 3


mode,Instruct,Reasoning
category,,
Small/Edge,15,5
Standard/Desktop,10,4
Tiny/Mobile,13,6


### 2. Inferences
We read the per-prompt records of every run of each execution (`results/main` and `results/mistral-bf16`), keeping the measurements of each inference, and the number of prompts that each run used to warm up the hardware, which is read from its log (`warmup_prompts`). Runs that were not executed are left out, and of each model run we keep the first repetitions that were executed, up to the number set for its execution (`REPETITIONS` in `common.py`). The answer extraction of `evaluate_from_local.py` is repeated to record which pattern gave each prediction (`pattern`: 1 for `the answer is (X)`, 2 for `Answer: X`, 3 for the last standalone letter, 0 for none), and whether the response closes its reasoning block (`reasoning_closed`).

In [3]:
CLOSING_TAG = re.compile(r"</think>|</reasoning>|<channel\|>|\[/THINK\]")
ANSWER_IS = re.compile(r"answer is \(?([A-J])\)?")
ANSWER_LINE = re.compile(r"[aA]nswer:\s*([A-J])")
STANDALONE_LETTER = re.compile(r"\b[A-J]\b")
# Progress bar that the log shows at the end of each group of warm-up prompts
WARMUP_BAR = re.compile(r"Processed prompts: 100%\|[^|]*\| (\d+)/\1 ")


def count_warmup_prompts(run_dir):
    # Prompts generated to warm up the hardware, before the inferences of the run start
    log = (run_dir / "experiment.log").read_text(errors="replace")
    warmup = log.split("Warmup completed, starting inference")[0]
    return sum(int(count) for count in WARMUP_BAR.findall(warmup))


def extract_answer(response):
    # The reasoning (everything up to the last closing tag) is removed first
    closing_tags = list(CLOSING_TAG.finditer(response))
    text = (response[closing_tags[-1].end():] if closing_tags else response).strip()

    match = ANSWER_IS.search(text)
    if match:
        return 1, match.group(1)

    first = ANSWER_LINE.search(text)
    if first:
        # Last match on the first line that has one
        line_start = text.rfind("\n", 0, first.start()) + 1
        line_end = text.find("\n", first.start())
        line_end = len(text) if line_end < 0 else line_end
        on_line = [m for m in ANSWER_LINE.finditer(text, line_start) if m.start() < line_end]
        return 2, on_line[-1].group(1)

    letters = STANDALONE_LETTER.findall(text)
    if letters:
        return 3, letters[-1]
    return 0, None


def load_execution(execution, results_dir):
    # The run table has the runs that were executed, in the order of execution: the first repetitions of each model run are kept
    run_table = load_run_table(execution)
    run_table = run_table[run_table.groupby("model").cumcount() < REPETITIONS[execution]].reset_index(drop=True)
    run_table["warmup_prompts"] = [count_warmup_prompts(results_dir / run_id) for run_id in run_table["__run_id"]]
    rows = []
    for _, run in run_table.iterrows():
        run_id = run["__run_id"]
        for path in sorted((results_dir / run_id).glob("*/CoT/all/*.json")):
            with open(path, "r") as f:
                records = json.load(f)
            for record in records:
                response = record["model_outputs"] or ""
                pattern, prediction = extract_answer(response)
                if prediction != record["pred"]:
                    raise ValueError(f"{run_id}, question {record['question_id']}: prediction not reproduced.")
                rows.append({
                    "execution": execution,
                    "run": run_id,
                    "model_run": run["model"],
                    "repetition": int(run_id.split("_")[-1]),
                    "question_id": record["question_id"],
                    "subject": record["category"],
                    "options": len(record["options"]),
                    "answer": record["answer"],
                    "prediction": prediction,
                    "pattern": pattern,
                    "reasoning_closed": bool(CLOSING_TAG.search(response)),
                    "prompt_tokens": record["prompt_tokens"],
                    "generated_tokens": record["generated_tokens"],
                    "thinking_tokens": record["thinking_tokens"],
                    "prefill_time_seconds": record["prefill_time_seconds"],
                    "decode_time_seconds": record["decode_time_seconds"],
                    "inference_time_seconds": record["inference_time_seconds"],
                    "gpu_joules": record["inference_gpu_joules"],
                    "cpu_joules": record["inference_cpu_joules"],
                    "dram_joules": record["inference_dram_joules"],
                    "gpu_temperature_celsius": record["gpu_temperature_celsius"],
                    "cpu_temperature_celsius": record["cpu_temperature_celsius"],
                })
    return run_table.assign(execution=execution), pd.DataFrame(rows)


run_tables, inference_tables = [], []
for execution, results_dir in RESULTS_DIRS.items():
    if not (results_dir / "run_table.csv").exists():
        print(f"[{execution}] no results yet in {results_dir.relative_to(ROOT_DIR)}")
        continue
    run_table, execution_inferences = load_execution(execution, results_dir)
    run_tables.append(run_table)
    inference_tables.append(execution_inferences)
    print(f"[{execution}] runs: {len(run_table)}   inferences: {len(execution_inferences)}")

run_table = pd.concat(run_tables, ignore_index=True)
inferences = pd.concat(inference_tables, ignore_index=True)

[main] runs: 530   inferences: 197160


[mistral-bf16] runs: 15   inferences: 5580


### 3. Inference Flags
We mark the correct predictions (`correct`), the responses that reached the limit of generated tokens (`truncated`), the reasoning responses that were cut off before the reasoning ended (`cut_off_reasoning`) and the correct predictions that were not truncated (`strict_correct`).

In [4]:
unknown = set(inferences["model_run"]) - set(model_runs["model_run"])
assert not unknown, f"Runs of models that are not in the selection: {unknown}"

mode = inferences["model_run"].map(model_runs.set_index("model_run")["mode"])
limit = MAX_NEW_TOKENS + np.where(mode == "Reasoning", MAX_THINKING_TOKENS, 0)
assert (inferences["generated_tokens"] <= limit).all()

inferences["correct"] = inferences["prediction"] == inferences["answer"]
inferences["truncated"] = inferences["generated_tokens"] >= limit
inferences["cut_off_reasoning"] = (mode == "Reasoning") & inferences["truncated"] & ~inferences["reasoning_closed"]
inferences["strict_correct"] = inferences["correct"] & ~inferences["truncated"]

missing = sorted(set(model_runs["model_run"]) - set(inferences["model_run"]))
model_runs["has_results"] = ~model_runs["model_run"].isin(missing)
print(f"Model runs without results: {missing if missing else 'none'}")

# Every repetition has all the questions
assert (inferences.groupby(["execution", "run"]).size() == QUESTIONS).all()
repetitions_per_run = inferences.groupby(["execution", "model_run"])["run"].nunique()
for execution, counts in repetitions_per_run.groupby("execution"):
    print(f"[{execution}] repetitions per model run: {counts.value_counts().sort_index().to_dict()} (number of repetitions: model runs)")
print(f"Inferences: {len(inferences)}   truncated: {inferences['truncated'].sum()}   without a prediction: {inferences['prediction'].isna().sum()}")

Model runs without results: none
[main] repetitions per model run: {10: 53} (number of repetitions: model runs)
[mistral-bf16] repetitions per model run: {5: 3} (number of repetitions: model runs)
Inferences: 202740   truncated: 12708   without a prediction: 2833


### 4. Repetitions
We join the run table of each execution with the counts of each repetition, and check that both agree.

In [5]:
repetitions = run_table.rename(columns={"__run_id": "run", "model": "model_run"}).drop(columns=["__done", "benchmark"])
repetitions.insert(2, "repetition", repetitions["run"].map(lambda run_id: int(run_id.split("_")[-1])))

counts = inferences.groupby(["execution", "run"]).agg(
    correct=("correct", "sum"),
    strict_correct=("strict_correct", "sum"),
    cut_off_reasoning=("cut_off_reasoning", "sum"),
    gpu_joules=("gpu_joules", "sum"),
    generated_tokens=("generated_tokens", "sum"),
    truncated=("truncated", "sum"),
    unparsed=("prediction", lambda x: x.isna().sum()),
).reset_index()
repetitions = repetitions.merge(counts, on=["execution", "run"], validate="one_to_one")

assert np.allclose(repetitions["gpu_joules"], repetitions["total_inference_gpu_joules"], rtol=1e-12)
assert (repetitions["generated_tokens"] == repetitions["total_generated_tokens"]).all()
assert (repetitions["truncated"] == repetitions["truncated_responses"]).all()
assert (repetitions["unparsed"] == repetitions["unparsed_responses"]).all()

repetitions["strict_accuracy"] = repetitions["strict_correct"] / QUESTIONS
repetitions["correct_answers_per_kj"] = repetitions["accuracy_per_gpu_joule"] * 1000
repetitions = repetitions.drop(columns=["gpu_joules", "generated_tokens", "truncated", "unparsed"])
repetitions = repetitions.rename(columns={
    "correct": "correct_predictions",
    "strict_correct": "strict_correct_predictions",
    "cut_off_reasoning": "cut_off_reasoning_responses",
})

print(f"Repetitions: {len(repetitions)}")

Repetitions: 545


### 5. Questions
We summarize the repetitions of each question of each model run: median and mean of its energy, tokens and time, and the share of repetitions in which it was correct, truncated or cut off.

In [6]:
questions = inferences.groupby(["model_run", "question_id"]).agg(
    subject=("subject", "first"),
    options=("options", "first"),
    gpu_joules_median=("gpu_joules", "median"),
    gpu_joules_mean=("gpu_joules", "mean"),
    generated_tokens_median=("generated_tokens", "median"),
    generated_tokens_mean=("generated_tokens", "mean"),
    prompt_tokens=("prompt_tokens", "first"),
    inference_time_seconds_median=("inference_time_seconds", "median"),
    inference_time_seconds_mean=("inference_time_seconds", "mean"),
    correct=("correct", "mean"),
    strict_correct=("strict_correct", "mean"),
    truncated=("truncated", "mean"),
    cut_off_reasoning=("cut_off_reasoning", "mean"),
).reset_index()

assert len(questions) == QUESTIONS * inferences["model_run"].nunique()
chance_level = (1 / questions.drop_duplicates("question_id")["options"]).mean()
print(f"Rows: {len(questions)}   questions: {questions['question_id'].nunique()}   chance level of the sample: {chance_level:.4f}")

Rows: 20832   questions: 372   chance level of the sample: 0.1138


### 6. Model Run Summary
We summarize each model run by the median of its repetitions, with their number (`repetitions`) and the minimum and maximum of the main metrics.

| Column | Meaning |
|---|---|
| `accuracy` | Accuracy as recorded by the script |
| `strict_accuracy` | Accuracy with every truncated response counted as wrong |
| `gpu_joules_per_question` | GPU energy of a repetition divided by the number of questions |
| `generated_tokens_per_question` | Generated tokens of a repetition divided by the number of questions |
| `gpu_joules_per_generated_token` | GPU energy divided by the generated tokens |
| `correct_answers_per_kj` | Correct answers per kJ of GPU energy (accuracy per joule × 1,000) |
| `gpu_joules_per_correct_answer` | GPU energy per correct answer |

In [7]:
MEDIANS = {
    "accuracy": "avg_accuracy",
    "strict_accuracy": "strict_accuracy",
    "gpu_joules_per_question": "avg_gpu_joules_per_inference",
    "median_gpu_joules_per_question": "median_gpu_joules_per_inference",
    "generated_tokens_per_question": "avg_generated_tokens_per_inference",
    "gpu_joules_per_generated_token": "gpu_joules_per_generated_token",
    "correct_answers_per_kj": "correct_answers_per_kj",
    "gpu_joules_per_correct_answer": "gpu_joules_per_correct_answer",
    "seconds_per_question": "avg_time_per_inference_seconds",
    "decode_tokens_per_second": "avg_decode_tokens_per_second",
    "gpu_power_watts": "avg_gpu_power_watts",
    "gpu_temperature_celsius": "avg_gpu_temperature_celsius",
    "truncated_responses": "truncated_responses",
    "unparsed_responses": "unparsed_responses",
    "cut_off_reasoning_responses": "cut_off_reasoning_responses",
    "thinking_tokens": "total_thinking_tokens",
    "prompt_tokens": "total_prompt_tokens",
}
WITH_RANGE = ["accuracy", "gpu_joules_per_question", "generated_tokens_per_question", "gpu_joules_per_generated_token", "correct_answers_per_kj"]

grouped = repetitions.groupby("model_run")

summary = pd.DataFrame({name: grouped[column].median() for name, column in MEDIANS.items()})
for name in WITH_RANGE:
    summary[f"{name}_min"] = grouped[MEDIANS[name]].min()
    summary[f"{name}_max"] = grouped[MEDIANS[name]].max()

# Number of repetitions and their totals
summary.insert(0, "repetitions", grouped.size())
summary["total_gpu_joules"] = grouped["total_inference_gpu_joules"].sum()
summary["total_inference_seconds"] = grouped["total_inference_time_seconds"].sum()
summary["total_runtime_seconds"] = grouped["runtime_seconds"].sum()
summary["execution"] = grouped["execution"].first()

model_runs = model_runs.merge(summary.reset_index(), on="model_run", how="left", validate="one_to_one")
model_runs["repetitions"] = model_runs["repetitions"].fillna(0).astype(int)
print(f"Model runs: {len(model_runs)}   with results: {model_runs['has_results'].sum()}")
model_runs[["model_run", "category", "mode", "precision", "in_study", "accuracy", "gpu_joules_per_question", "correct_answers_per_kj"]].head(10)

Model runs: 56   with results: 56


,model_run,category,mode,precision,in_study,accuracy,gpu_joules_per_question,correct_answers_per_kj
0,Qwen/Qwen1.5-4B-Chat,Tiny/Mobile,Instruct,16-bit,True,0.236559,465.602407,0.508182
1,google/gemma-1.1-2b-it,Tiny/Mobile,Instruct,16-bit,True,0.180108,184.130867,0.988220
2,Qwen/Qwen2-1.5B-Instruct,Tiny/Mobile,Instruct,16-bit,True,0.236559,197.146276,1.210410
3,google/gemma-2-2b-it,Tiny/Mobile,Instruct,16-bit,True,0.284946,339.118806,0.835412
4,microsoft/Phi-3.5-mini-instruct,Tiny/Mobile,Instruct,16-bit,True,0.427419,884.722224,0.483111
5,Qwen/Qwen2.5-3B-Instruct,Tiny/Mobile,Instruct,16-bit,True,0.465054,752.127039,0.618318
6,meta-llama/Llama-3.2-3B-Instruct,Tiny/Mobile,Instruct,16-bit,True,0.364247,580.000772,0.635983
7,deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B,Tiny/Mobile,Reasoning,16-bit,True,0.392473,3974.782046,0.098741
8,microsoft/Phi-4-mini-instruct,Tiny/Mobile,Instruct,16-bit,True,0.470430,564.453073,0.833427
9,google/gemma-3-4b-it,Tiny/Mobile,Instruct,16-bit,True,0.459677,1232.213862,0.373050


### 7. Validation
We check that accuracy per joule equals accuracy ÷ (generated tokens per question × energy per generated token), that the totals of the main execution are the expected ones, and that the table of questions gives back the energy per question of each model run.

In [8]:
identity = repetitions["avg_accuracy"] / (repetitions["avg_generated_tokens_per_inference"] * repetitions["gpu_joules_per_generated_token"])
assert np.allclose(identity, repetitions["accuracy_per_gpu_joule"], rtol=1e-9)

main = inferences[inferences["execution"] == "main"]
assert len(main) == 197160 and main["truncated"].sum() == 12288 and main["prediction"].isna().sum() == 2731
assert main["cut_off_reasoning"].sum() == 7959

from_questions = questions.groupby("model_run")["gpu_joules_mean"].mean()
from_repetitions = repetitions.groupby("model_run")["avg_gpu_joules_per_inference"].mean()
assert np.allclose(from_questions, from_repetitions.loc[from_questions.index], rtol=1e-9)

print("All checks passed.")

All checks passed.


### 8. Saving the Tables
The tables are saved to `analysis/data/`: one row per inference, per repetition, per model run and question, and per model run.

In [9]:
DATA_DIR.mkdir(exist_ok=True)

inferences.to_csv(DATA_DIR / "inferences.csv", index=False)
repetitions.to_csv(DATA_DIR / "repetitions.csv", index=False)
questions.to_csv(DATA_DIR / "questions.csv", index=False)
model_runs.to_csv(DATA_DIR / "model_runs.csv", index=False)

print(f"inferences: {inferences.shape}   repetitions: {repetitions.shape}   questions: {questions.shape}   model runs: {model_runs.shape}")

inferences: (202740, 26)   repetitions: (545, 43)   questions: (20832, 15)   model runs: (56, 46)
